# Nitrogen pulse inconsistency review — upstream kinetic model vs CO2 pipeline

**Diagnostic audit — LAB004–LAB012 (LAB009 included). No recalibration, no parameter changes, no fixes.**

Central question: **¿por qué el mismo experimento termina teniendo tiempos y/o dosis de pulso de nitrógeno diferentes según el pipeline que lo procesa?**

The notebook compares the two pulse events actually consumed by each pipeline:

| Pipeline | Time source | Dose source |
| --- | --- | --- |
| **Upstream kinetic model** (`historical.make_medium_batches("natural")` → loader) | timestamp (`t` column) of the workbook `pulso_nut` row in `mosto_natural_xthiol.xlsx` | value recorded in that row (80 mg/L = 0.08 kg/m³) |
| **CO2 layer / pipeline** (`natural_nutrient_pulse_schedule` + `override_natural_nutrient_pulses`) | density-1040 crossing interpolated from chemistry, exposed as `timing_source`; ICS calendar only as fallback | protocol constant derived from `SPRINGFERM_XTREM_G` + `FDA_G` + `YAN_MG_PER_MG_PRODUCT` (0.14 kg/m³), applied to every natural batch |

The **ICS calendar is documented as traceability/fallback only** (`calendar_t_h`): when a 1040 crossing exists it is not used, and no third pulse line is drawn in the figures.

Constraints honored in this notebook: LAB009 is included as a temporal reference even though its CO2 was excluded from the historical CO2 calibration by QC (documented, not hidden); frozen CO2 predictions are loaded read-only from the existing canonical run; nothing is refitted; no external CSV/PNG artifacts are written — tables and figures live only in this notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next(path for path in candidates if (path / "fermentation_model").exists())
FM = ROOT / "fermentation_model"
if str(FM) not in sys.path:
    sys.path.insert(0, str(FM))

from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross
from laboratory_2026 import run_estimability_historical_by_medium as historical
from shared import run_new_must_glycerol_estimability_doe as kinetic

LABS = [f"LAB{number:03d}" for number in range(4, 13)]
WORKBOOK = FM / "data" / "Laboratorio 2026" / "Vendimia_2026" / "mosto_natural_xthiol.xlsx"
NORM_CACHE = FM / "shared" / "results" / "new_must_data_loading" / "new_must_normalized_long.csv"
THETA_FULL_PATH = FM / "laboratory_2026" / "results" / "estimability_historical_natural" / "theta_natural_full.csv"
CO2_RESULTS = FM / "laboratory_2026" / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010"
EXCLUDED_PATH = CO2_RESULTS / "excluded_experiments.csv"

pd.set_option("display.width", 200)
print("Repository root:", ROOT)
print("Workbook:", WORKBOOK.name)
print("Frozen CO2 run (read-only provenance):", CO2_RESULTS.name)

## 1. Auditoría del origen temporal (`t=0`)

Antes de comparar tiempos de pulso se verifica que ambos pipelines usen el **mismo origen**. El loader upstream y el runner CO2 obtienen `t=0` de la misma función (`load_natural_batch_metadata`), que lee la columna `fecha_hora` de la fila `t == 0` de cada hoja del workbook. Aquí se re-verifica contra una lectura directa del workbook.

Referencia de código: `run_estimability_historical_by_medium.py::load_natural_batch_metadata` (filas `t == 0.0` → `fecha_hora`); el runner CO2 consume esa misma metadata en `natural_nutrient_pulse_schedule(..., metadata)`.

In [ ]:
raw_t0 = {}
raw_t0_note = {}
for lab in LABS:
    sheet = pd.read_excel(WORKBOOK, sheet_name=lab)
    sheet["t_num"] = pd.to_numeric(sheet["t"], errors="coerce")
    zero = sheet[sheet["t_num"].eq(0.0)]
    assert not zero.empty, f"No t=0 row in {lab}"
    raw_t0[lab] = pd.to_datetime(zero["fecha_hora"].iloc[0])
    raw_t0_note[lab] = str(zero["fecha_hora_nota"].iloc[0]) if "fecha_hora_nota" in zero.columns else ""

metadata = historical.load_natural_batch_metadata(LABS)
t0_audit = metadata[["batch", "t0"]].copy()
t0_audit["workbook_t0_fecha_hora"] = t0_audit["batch"].map(raw_t0)
t0_audit["fecha_hora_nota"] = t0_audit["batch"].map(raw_t0_note)
t0_audit["match"] = [
    pd.Timestamp(a) == pd.Timestamp(b)
    for a, b in zip(t0_audit["t0"], t0_audit["workbook_t0_fecha_hora"])
]
display(t0_audit)
assert t0_audit["match"].all(), "t0 mismatch between workbook and loader metadata"
print("t=0 origin is identical for both pipelines (loader metadata == workbook t=0 row).")
print("Therefore the pulse-time discrepancy is NOT a time-origin problem.")

## 2. Procedencia del pulso upstream (columna `pulso_nut` del workbook → loader → `batch.pulses`)

El loader lee la columna `pulso_nut` como **dosis del pulso en mg/L** y usa la columna `t` de esa misma fila como **tiempo del pulso**:

- `new_must_data_loader.py`: `out["N_pulse_mg_l"] = _first_available_numeric(raw, ["pulso_nut"])`, `out["time_h"] = _first_available_numeric(raw, ["time", "t", "Horas"])`.
- `batch.pulses["N"] = ((t_fila, dosis_mg_l × 1e-3),)` — un único pulso en proceso por lote.

La columna `fecha_hora` de esas filas se muestra como trazabilidad; varias son seriales Excel (`fecha_hora_nota = serial_excel`) y **no** son la fuente del tiempo upstream.

In [ ]:
def as_timestamp(value):
    if isinstance(value, (int, float)) and not pd.isna(value):
        return pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(value))
    return pd.to_datetime(value, errors="coerce")

raw_rows = []
for lab in LABS:
    sheet = pd.read_excel(WORKBOOK, sheet_name=lab)
    pulse = pd.to_numeric(sheet["pulso_nut"], errors="coerce")
    mask = pulse.fillna(0.0) > 0.0
    for idx, row in sheet[mask].iterrows():
        raw_rows.append({
            "batch": lab,
            "sample_id": row["ID"],
            "row_fecha_hora_raw": row["fecha_hora"],
            "row_fecha_hora_decoded": as_timestamp(row["fecha_hora"]),
            "t_h_workbook": float(row["t"]),
            "pulso_nut_mg_l": float(row["pulso_nut"]),
        })
upstream_raw = pd.DataFrame(raw_rows)

norm = pd.read_csv(NORM_CACHE)
norm_pulse = norm[norm["N_pulse_mg_l"].fillna(0.0) > 0.0][["batch", "sample_id", "time_h", "N_pulse_mg_l"]]

data, batches, _co2_unused, _process = historical.make_medium_batches("natural")
live = {}
for batch in batches:
    pulses = batch.pulses.get("N", tuple())
    assert len(pulses) == 1, f"{batch.batch}: expected exactly one in-process N pulse"
    live[batch.batch] = pulses[0]

upstream = upstream_raw.copy()
upstream["time_h_loader_cache"] = upstream["batch"].map(norm_pulse.set_index("batch")["time_h"])
upstream["dose_mg_l_loader_cache"] = upstream["batch"].map(norm_pulse.set_index("batch")["N_pulse_mg_l"])
upstream["t_pulse_upstream_h"] = upstream["batch"].map(lambda b: live[b][0])
upstream["deltaN_upstream_kg_m3"] = upstream["batch"].map(lambda b: live[b][1])
upstream["match_cache"] = [
    bool(np.isclose(a, b) and np.isclose(c, d))
    for a, b, c, d in zip(
        upstream["t_h_workbook"], upstream["time_h_loader_cache"],
        upstream["pulso_nut_mg_l"], upstream["dose_mg_l_loader_cache"],
    )
]
upstream["match_live_batches"] = [
    bool(np.isclose(r.t_h_workbook, r.t_pulse_upstream_h) and np.isclose(r.pulso_nut_mg_l * 1e-3, r.deltaN_upstream_kg_m3))
    for r in upstream.itertuples()
]
display(upstream)
assert upstream["match_cache"].all() and upstream["match_live_batches"].all()
print("Upstream pulse = workbook `pulso_nut` row: dose 80 mg/L -> 0.08 kg/m3, time = row `t`.")

## 3. Procedencia del pulso CO2 (calendario ICS → cruce de densidad 1040 → dosis de protocolo)

El runner CO2 construye el pulso en `natural_nutrient_pulse_schedule`:

1. Enumera los eventos `Pulso nutricional 2` del **ICS** (`_calendar_nutrient_pulse_events`) — el calendario aporta el evento y su `calendar_t_h` (trazabilidad/fallback).
2. **Sobrescribe el tiempo** con el cruce de densidad 1040 interpolado desde la química (`_density_crossing`), siempre que exista, *"even when it is wide"*; el calendario solo actúa si la química nunca cruza 1040.
3. Fija la **dosis** con una constante de protocolo (`SPRINGFERM_XTREM_G` + `FDA_G` + `YAN_MG_PER_MG_PRODUCT`, reactor 2 L), no con el valor de la planilla.
4. `override_natural_nutrient_pulses` inyecta ese tiempo/dosis en los batches que consume la capa CO2.

In [ ]:
calendar = co2_cross._calendar_nutrient_pulse_events(co2_cross.NUTRIENT_CALENDAR_PATH)
print("ICS in-process pulse events (Pulso nutricional 2):")
display(calendar[["batch", "calendar_timestamp_local", "calendar_product", "calendar_dose", "calendar_process_state"]])

chem = norm[["batch", "time_h", "density"]].copy()
schedule = co2_cross.natural_nutrient_pulse_schedule(chem, metadata)
schedule_cols = [
    "batch", "calendar_t_h", "density_crossing_t_h", "density_bracket_start_h",
    "density_bracket_end_h", "density_bracket_width_h", "model_pulse_time_h",
    "timing_source", "calendar_minus_model_h", "amount_N_kg_m3",
]
display(schedule[schedule_cols])

print("CO2 pipeline protocol dose constant:")
print("  SPRINGFERM_XTREM_G =", co2_cross.SPRINGFERM_XTREM_G, "g")
print("  FDA_G =", co2_cross.FDA_G, "g")
print("  YAN_MG_PER_MG_PRODUCT =", co2_cross.YAN_MG_PER_MG_PRODUCT, "mg YAN per mg product")
amount_value = float(schedule["amount_N_kg_m3"].iloc[0])
print("  -> amount_N_kg_m3 applied to every natural batch =", round(amount_value, 6))

### 3.1 Verificación contra el artefacto canónico de la corrida congelada

La corrida histórica congelada (`co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010`) dejó su propia tabla de pulsos efectivos (`effective_nutrient_pulses.csv`). Se compara la recomputación de arriba contra ese artefacto para probar que esta auditoría reproduce exactamente lo que la capa CO2 consumió.

In [ ]:
artifact = pd.read_csv(CO2_RESULTS / "effective_nutrient_pulses.csv")
merged = schedule.merge(
    artifact[["batch", "pulse_time_h", "amount_N_kg_m3", "timing_source"]],
    on="batch", suffixes=("_recomputed", "_artifact"),
)
merged["time_match"] = np.isclose(merged["model_pulse_time_h"], merged["pulse_time_h"])
merged["amount_match"] = np.isclose(merged["amount_N_kg_m3_recomputed"], merged["amount_N_kg_m3_artifact"])
merged["source_match"] = merged["timing_source_recomputed"].eq(merged["timing_source_artifact"])
display(merged[[
    "batch", "model_pulse_time_h", "pulse_time_h",
    "amount_N_kg_m3_recomputed", "amount_N_kg_m3_artifact",
    "timing_source_recomputed", "timing_source_artifact",
    "time_match", "amount_match", "source_match",
]])
assert merged[["time_match", "amount_match", "source_match"]].all().all()
print("Recomputed schedule == frozen-run effective pulses (time, dose and timing_source).")

## 4. Tabla principal — pulso upstream vs pulso CO2

`delta_t_h = t_pulse_CO2 − t_pulse_upstream` (negativo ⇒ el pulso CO2 está antes que el upstream). `calendar_t_h` se muestra solo como trazabilidad/fallback, no como un tercer pulso.

In [ ]:
qc_reason = {}
if EXCLUDED_PATH.exists():
    excl = pd.read_csv(EXCLUDED_PATH)
    qc_reason = excl[excl["matrix"].eq("natural")].set_index("batch")["reason"].to_dict()

sch_idx = schedule.set_index("batch")
rows = []
for lab in LABS:
    u_t, u_dose = live[lab]
    s = sch_idx.loc[lab]
    notes = []
    if lab in qc_reason:
        notes.append("CO2 QC-excluded historically: " + str(qc_reason[lab]))
    if s["density_bracket_width_h"] > co2_cross.MAX_DENSITY_INTERPOLATION_BRACKET_H:
        notes.append("wide density bracket (sparse chemistry)")
    rows.append({
        "batch": lab,
        "t_pulse_upstream_h": float(u_t),
        "deltaN_upstream_kg_m3": float(u_dose),
        "source_upstream": "workbook `pulso_nut` row (sheet column `t`)",
        "t_pulse_CO2_h": float(s["model_pulse_time_h"]),
        "deltaN_CO2_kg_m3": float(s["amount_N_kg_m3"]),
        "source_CO2": str(s["timing_source"]),
        "delta_t_h_CO2_minus_upstream": float(s["model_pulse_time_h"]) - float(u_t),
        "dose_ratio_CO2_over_upstream": float(s["amount_N_kg_m3"]) / float(u_dose),
        "density_bracket_width_h": float(s["density_bracket_width_h"]),
        "calendar_t_h_traceability": float(s["calendar_t_h"]),
        "notes": "; ".join(notes),
    })
summary = pd.DataFrame(rows)
display(summary)

print("delta_t_h range:", round(summary["delta_t_h_CO2_minus_upstream"].min(), 2), "to", round(summary["delta_t_h_CO2_minus_upstream"].max(), 2), "h")
print("Batches with |delta_t| < 1 h:", summary.loc[summary["delta_t_h_CO2_minus_upstream"].abs() < 1.0, "batch"].tolist())
print("Dose ratio unique values:", sorted(set(np.round(summary["dose_ratio_CO2_over_upstream"], 4))))

## 5. Datos para las figuras (solo lectura, congelados)

- **CO2 observado y predicción congelada**: `prediction_rows.csv` de la corrida histórica canónica (matriz natural → natural). LAB009 no tiene predicción congelada (fue excluido por QC de la calibración CO2 histórica) y se muestra solo observado, con anotación.
- **N observado**: columna `N_kg_m3` del cache del loader (YAN medido; las filas de pulso `L-N` no llevan medición y quedan fuera por `dropna`).
- **N simulado upstream**: `kinetic.simulate(batch, theta_full, batch.time)` con `theta_natural_full.csv` (17 parámetros canónicos), sobre la grilla canónica de cada lote.

In [ ]:
pred = pd.read_csv(CO2_RESULTS / "prediction_rows.csv")
nat = pred[pred["calibration_matrix"].eq("natural") & pred["target_matrix"].eq("natural")].copy()
nat = nat.sort_values(["batch", "time_h"])
print("Frozen CO2 model:", nat["model"].unique().tolist())
print("Frozen CO2 batches:", sorted(nat["batch"].unique()))

theta_frame = pd.read_csv(THETA_FULL_PATH)
theta_full = theta_frame.set_index("parameter")["theta"].astype(float).to_dict()
assert tuple(theta_frame["parameter"]) == tuple(kinetic.FULL17)
print("theta_full: 17-parameter canonical vector loaded from", THETA_FULL_PATH.name)

n_obs = norm.dropna(subset=["N_kg_m3"])[["batch", "time_h", "N_kg_m3", "YAN_mg_l"]].copy()
n_obs = n_obs[n_obs["batch"].isin(LABS)]

trajectories = {b.batch: kinetic.simulate(b, theta_full, b.time) for b in batches}
assert all(frame is not None for frame in trajectories.values())
print("Upstream N trajectories simulated for:", sorted(trajectories))

## 6. Figura combinada CO2 + N (un subplot por LAB)

Eje izquierdo: CO2 observado (negro) y predicción congelada (gris) en g/L/h. Eje derecho: N simulado upstream (azul) y N/YAN observado (puntos azules) en kg/m³. Línea vertical **verde continua** = pulso usado por upstream (`pulso_nut`); línea vertical **roja discontinua** = pulso efectivo usado por la capa CO2 (cruce 1040). LAB009 aparece sin predicción congelada y con anotación de su exclusión histórica por QC.

In [ ]:
CO2_OBS = "0.15"
CO2_PRED = "0.55"
N_MODEL = "#1f77b4"
UP_STYLE = dict(color="#2ca02c", lw=1.8, ls="-")
CO2P_STYLE = dict(color="#d62728", lw=1.8, ls="--")
UP_LABEL = "pulse used by upstream (`pulso_nut` row)"
CO2P_LABEL = "effective pulse used by CO2 pipeline (density-1040 crossing)"

def draw_pulse_lines(ax, lab):
    ax.axvline(live[lab][0], **UP_STYLE, label=UP_LABEL)
    ax.axvline(float(sch_idx.loc[lab]["model_pulse_time_h"]), **CO2P_STYLE, label=CO2P_LABEL)

def combined_figure():
    fig, axes = plt.subplots(3, 3, figsize=(16.5, 11.5))
    reference_handles = None
    for ax, lab in zip(axes.flat, LABS):
        g = nat[nat["batch"].eq(lab)]
        if not g.empty:
            ax.plot(g["time_h"], g["observed_g_l_h"], color=CO2_OBS, lw=1.0, label="CO2 observed (SCCM-corrected)")
            ax.plot(g["time_h"], g["predicted_g_l_h"], color=CO2_PRED, lw=1.4, label="CO2 frozen prediction (historical run)")
        ax.set_xlabel("time from historical batch origin [h]")
        ax.set_ylabel("CO2 [g/L/h]")
        ax2 = ax.twinx()
        frame = trajectories[lab]
        ax2.plot(frame.index.to_numpy(dtype=float), frame["N"].to_numpy(dtype=float), color=N_MODEL, lw=1.6, label="N simulated (upstream, theta_full)")
        obs = n_obs[n_obs["batch"].eq(lab)]
        ax2.plot(obs["time_h"], obs["N_kg_m3"], "o", ms=4, color=N_MODEL, alpha=0.8, label="N/YAN observed")
        ax2.set_ylabel("N [kg/m3]")
        draw_pulse_lines(ax, lab)
        delta = float(sch_idx.loc[lab]["model_pulse_time_h"]) - live[lab][0]
        extra = "  —  CO2 QC-excluded in historical calibration" if lab == "LAB009" else ""
        ax.set_title(f"{lab}   (delta_t = {delta:+.2f} h){extra}", fontsize=10)
        if reference_handles is None:
            h1, l1 = ax.get_legend_handles_labels()
            h2, l2 = ax2.get_legend_handles_labels()
            reference_handles = dict(zip(l1 + l2, h1 + h2))
    fig.suptitle("Nitrogen pulse inconsistency — CO2 observed/frozen vs N simulated/observed (LAB004-LAB012, incl. LAB009)", fontsize=13)
    order = [
        "CO2 observed (SCCM-corrected)", "CO2 frozen prediction (historical run)",
        "N simulated (upstream, theta_full)", "N/YAN observed", UP_LABEL, CO2P_LABEL,
    ]
    fig.legend([reference_handles[k] for k in order], order, loc="lower center", bbox_to_anchor=(0.5, -0.015), ncol=3, frameon=False, fontsize=9)
    fig.tight_layout(rect=(0.0, 0.045, 1.0, 0.97))
    return fig

combined_figure()
plt.show()

## 7. Figura global de CO2 (referencia temporal del proceso)

CO2 observado vs predicción congelada, con ambos pulsos marcados. LAB009 se mantiene visible con su anotación de exclusión histórica; su curva se usa aquí solo como referencia temporal, no para reevaluar calidad de ajuste ni reabrir la selección de batches.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16.5, 11.0))
reference_handles = None
for ax, lab in zip(axes.flat, LABS):
    g = nat[nat["batch"].eq(lab)]
    if not g.empty:
        ax.plot(g["time_h"], g["observed_g_l_h"], color=CO2_OBS, lw=1.0, label="CO2 observed (SCCM-corrected)")
        ax.plot(g["time_h"], g["predicted_g_l_h"], color=CO2_PRED, lw=1.4, label="CO2 frozen prediction (historical run)")
    ax.set_xlabel("time from historical batch origin [h]")
    ax.set_ylabel("CO2 [g/L/h]")
    draw_pulse_lines(ax, lab)
    delta = float(sch_idx.loc[lab]["model_pulse_time_h"]) - live[lab][0]
    extra = "  —  CO2 QC-excluded in historical calibration" if lab == "LAB009" else ""
    ax.set_title(f"{lab}   (delta_t = {delta:+.2f} h){extra}", fontsize=10)
    if reference_handles is None:
        h1, l1 = ax.get_legend_handles_labels()
        reference_handles = dict(zip(l1, h1))
fig.suptitle("CO2 observed vs frozen prediction with both nitrogen pulse conventions (LAB004-LAB012, incl. LAB009)", fontsize=13)
order = ["CO2 observed (SCCM-corrected)", "CO2 frozen prediction (historical run)", UP_LABEL, CO2P_LABEL]
fig.legend([reference_handles[k] for k in order], order, loc="lower center", bbox_to_anchor=(0.5, -0.015), ncol=2, frameon=False, fontsize=9)
fig.tight_layout(rect=(0.0, 0.045, 1.0, 0.97))
plt.show()

## 8. Figura global de N

N simulado upstream (con su salto de `pulso_nut`) y observaciones YAN, con ambos pulsos marcados. Permite ver: agotamiento de N, posición del salto upstream, posición del pulso CO2 y su distancia respecto del salto, y el nivel de N en el momento de cada pulso.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16.5, 11.0))
reference_handles = None
for ax, lab in zip(axes.flat, LABS):
    frame = trajectories[lab]
    ax.plot(frame.index.to_numpy(dtype=float), frame["N"].to_numpy(dtype=float), color=N_MODEL, lw=1.6, label="N simulated (upstream, theta_full)")
    obs = n_obs[n_obs["batch"].eq(lab)]
    ax.plot(obs["time_h"], obs["N_kg_m3"], "o", ms=4, color=N_MODEL, alpha=0.8, label="N/YAN observed")
    ax.set_xlabel("time from historical batch origin [h]")
    ax.set_ylabel("N [kg/m3]")
    draw_pulse_lines(ax, lab)
    delta = float(sch_idx.loc[lab]["model_pulse_time_h"]) - live[lab][0]
    ax.set_title(f"{lab}   (delta_t = {delta:+.2f} h)", fontsize=10)
    if reference_handles is None:
        h1, l1 = ax.get_legend_handles_labels()
        reference_handles = dict(zip(l1, h1))
fig.suptitle("Nitrogen trajectory and observations with both pulse conventions (LAB004-LAB012, incl. LAB009)", fontsize=13)
order = ["N simulated (upstream, theta_full)", "N/YAN observed", UP_LABEL, CO2P_LABEL]
fig.legend([reference_handles[k] for k in order], order, loc="lower center", bbox_to_anchor=(0.5, -0.015), ncol=2, frameon=False, fontsize=9)
fig.tight_layout(rect=(0.0, 0.045, 1.0, 0.97))
plt.show()

## 9. Conclusiones (basadas exclusivamente en la evidencia verificada arriba)

1. **¿En qué LAB coinciden o difieren los tiempos?** Ninguno coincide exactamente. LAB009 es el más cercano (+0.85 h). Los ocho restantes difieren: el pulso CO2 está **antes** que el upstream en abril (LAB004 −11.91, LAB005 −12.91, LAB006 −25.00, LAB007 −8.75, LAB008 −8.25 h) y **después** en mayo (LAB010 +23.49, LAB011 +14.84, LAB012 +19.04 h).

2. **¿Rango de las diferencias temporales?** −25.00 h a +23.49 h (|Δt| de 0.85 a 25.0 h). El transitorio completo de N tras el pulso dura ~5 h, así que la brecha excede el propio evento en la mayoría de los lotes.

3. **¿La inconsistencia es de tiempo, dosis o ambas?** **Ambas, en los nueve lotes.** Tiempo: como arriba. Dosis: upstream 0.08 kg/m³ (80 mg/L anotados en `pulso_nut`) vs capa CO2 0.14 kg/m³ (constante de protocolo), ratio 1.75 en todos.

4. **¿Por qué cada pipeline termina usando una referencia distinta?** Por diseño de cada pipeline, no por un bug de ejecución: el loader upstream consume la columna operacional `pulso_nut` (tiempo = `t` de la fila, dosis = valor de la fila); el runner CO2 **reemplaza deliberadamente** ese pulso (`override_natural_nutrient_pulses`) por una reconstrucción funcional — el instante en que la densidad cruzó 1040 g/L — y una dosis de protocolo uniforme. Los dos pipelines responden a definiciones distintas de "cuándo/cuánto fue el pulso".

5. **¿En qué LAB el tiempo CO2 viene del cruce 1040 y en cuáles de un fallback?** En los nueve lotes el tiempo efectivo viene del **cruce 1040**: `chemical_density_linear_interpolation` en LAB004–009 (brackets 0–17 h) y `chemical_density_linear_interpolation_sparse_bracket` en LAB010–012 (brackets 67–72 h). El fallback de calendario (`calendar_timestamp_missing_density_crossing`) no se usó en ningún lote; `calendar_t_h` quedó como trazabilidad.

6. **¿Qué tan confiable es el cruce 1040 cuando el bracket es muy ancho?** Baja en LAB010–012: interpolar linealmente un cruce entre muestras separadas 67–72 h (densidad en desaceleración no lineal) produce una incertidumbre del orden de días, comparable a la brecha misma (+14.8 a +23.5 h vs hoja/plan, que en mayo sí coinciden entre sí a <0.3 h). La propia corrida los marca `sparse_bracket`; deben tratarse como tiempo débilmente identificado.

7. **¿Cómo puede afectar esto la interpretación de la respuesta post-pulso de CO2?** El modelo de boost de N (rampa de utilización + ganancia de actividad) arranca en el tiempo efectivo del pipeline CO2. Con el pulso mal ubicado, la reactivación simulada se desalinea de la real: en mayo llega 15–23 h tarde (consistente con el exceso de CO2 observado sobre el congelado reportado para LAB011/LAB012 en el diagnóstico de microfugas) y en abril llega ~9–25 h antes. Además la dosis 1.75× escala el incremento upstream del contrafactual. Ambos efectos se compensan en el ajuste a través de `pulse_activity_gain`/`pulse_t_rise_h`, por lo que esos parámetros heredan el sesgo de tiempo/dosis documentado aquí.

8. **¿Qué debe unificarse antes de estudiar `pulse_activity_gain`, rise/decay o recalibrar el nitrógeno?** Un **ledger único de pulsos por lote** (tiempo + dosis + fuente + incertidumbre) aceptado por ambos pipelines, con decisión explícita de: (a) qué convención temporal es la referencia (con dosificación manual, la fila de visita plausiblemente es el acto real en abril, mientras que el cruce disperso de mayo es débil); (b) qué dosis corresponde por campaña (80 ppm registradas vs 140 ppm de protocolo); y (c) cómo se declara la incertidumbre (ancho de bracket). Sin esa unificación, cualquier parámetro de respuesta al N ajustado sobre estos datos hereda la inconsistencia caracterizada en este notebook.

In [ ]:
import scipy

print("Reproducibility:")
print("  python", sys.version.split()[0])
print("  pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
print("  workbook:", WORKBOOK.name)
print("  loader cache:", NORM_CACHE.name)
print("  theta:", THETA_FULL_PATH.name)
print("  frozen CO2 run:", CO2_RESULTS.name)
print("External artifacts written by this notebook: none (tables and figures are in-notebook only).")
print("No recalibration, no parameter changes, no pulse fixes were performed.")